# 22 - Optimize, Z-Order, Vacuum, and File Metrics

Optimizes two frequently accessed Silver tables and persists
before/after active-file metrics.

- `silver_events` Z-Orders by `user_id, event_id` for behavioral
  joins and dedup/lookups.
- `silver_orders` Z-Orders by `user_id, order_id` for Customer
  360 joins and SCD1 MERGE/lookups.
- VACUUM retention is **720 hours (30 days)**. Schedule this
  notebook weekly with `vacuum_mode=EXECUTE` only after Time
  Travel/CDF evidence is captured. The default `DRY_RUN` protects
  historical files while showing what would be removed.

Thirty days balances storage cleanup with the ability to debug,
replay CDF, and query historical Delta versions. VACUUM permanently
removes files older than the retention boundary.


In [ ]:
%run ./05_bronze_common


In [ ]:
import uuid

from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql import types as T


dbutils.widgets.text("catalog", "abc_retail", "Target catalog")
dbutils.widgets.text("job_run_id", "manual", "Notebook or job run ID")
dbutils.widgets.dropdown(
    "vacuum_mode", "DRY_RUN", ["DRY_RUN", "EXECUTE", "SKIP"], "Vacuum mode"
)

catalog = validate_identifier(dbutils.widgets.get("catalog").strip(), "catalog")
job_run_id = dbutils.widgets.get("job_run_id").strip()
vacuum_mode = dbutils.widgets.get("vacuum_mode").strip().upper()
spark.conf.set("spark.sql.session.timeZone", "UTC")

pipeline_name = "delta_optimization"
pipeline_run_id = str(uuid.uuid4())
start_ts = datetime.now(timezone.utc)
metrics_table = f"{catalog}.control.delta_optimization_metrics"
retention_hours = 720
targets = [
    (
        f"{catalog}.silver.silver_events",
        ["user_id", "event_id"],
        "user_id supports behavioral/customer joins; event_id supports dedup lookup",
    ),
    (
        f"{catalog}.silver.silver_orders",
        ["user_id", "order_id"],
        "user_id supports customer joins; order_id supports MERGE and lookup",
    ),
]
source_tables = [table for table, _, _ in targets]

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS `{catalog}`.`control`.`delta_optimization_metrics` (
        table_name STRING NOT NULL,
        metric_stage STRING NOT NULL,
        observed_ts TIMESTAMP NOT NULL,
        table_version BIGINT NOT NULL,
        num_files BIGINT NOT NULL,
        size_in_bytes BIGINT NOT NULL,
        avg_file_size_bytes DOUBLE NOT NULL,
        zorder_columns ARRAY<STRING> NOT NULL,
        zorder_reason STRING NOT NULL,
        vacuum_mode STRING NOT NULL,
        vacuum_retention_hours BIGINT NOT NULL,
        pipeline_run_id STRING NOT NULL
    )
    USING DELTA
    TBLPROPERTIES (
        'comment' = 'Before and after file metrics for Delta optimization'
    )
    """
)

metric_schema = T.StructType(
    [
        T.StructField("table_name", T.StringType(), False),
        T.StructField("metric_stage", T.StringType(), False),
        T.StructField("observed_ts", T.TimestampType(), False),
        T.StructField("table_version", T.LongType(), False),
        T.StructField("num_files", T.LongType(), False),
        T.StructField("size_in_bytes", T.LongType(), False),
        T.StructField("avg_file_size_bytes", T.DoubleType(), False),
        T.StructField("zorder_columns", T.ArrayType(T.StringType()), False),
        T.StructField("zorder_reason", T.StringType(), False),
        T.StructField("vacuum_mode", T.StringType(), False),
        T.StructField("vacuum_retention_hours", T.LongType(), False),
        T.StructField("pipeline_run_id", T.StringType(), False),
    ]
)


def quoted_table(table):
    return ".".join(f"`{part}`" for part in table.split("."))


def capture_metric(table, stage, columns, reason):
    detail = spark.sql(f"DESCRIBE DETAIL {quoted_table(table)}").first()
    version = int(
        spark.sql(f"DESCRIBE HISTORY {quoted_table(table)}")
        .agg(F.max("version").alias("version"))
        .first()
        .version
    )
    num_files = int(detail["numFiles"])
    size_in_bytes = int(detail["sizeInBytes"])
    avg_size = float(size_in_bytes / num_files) if num_files else 0.0
    return (
        table,
        stage,
        datetime.now(timezone.utc),
        version,
        num_files,
        size_in_bytes,
        avg_size,
        columns,
        reason,
        vacuum_mode,
        retention_hours,
        pipeline_run_id,
    )


upsert_audit(
    catalog, pipeline_run_id, pipeline_name, "optimize_zorder", start_ts,
    "RUNNING", source_tables, job_run_id,
)

try:
    metrics = []
    for table, columns, reason in targets:
        if not spark.catalog.tableExists(table):
            raise RuntimeError(f"Missing optimization target: {table}")

        metrics.append(capture_metric(table, "before", columns, reason))
        zorder = ", ".join(f"`{column}`" for column in columns)
        spark.sql(
            f"OPTIMIZE {quoted_table(table)} ZORDER BY ({zorder})"
        ).collect()
        metrics.append(capture_metric(table, "after", columns, reason))

        if vacuum_mode == "DRY_RUN":
            eligible = spark.sql(
                f"""
                VACUUM {quoted_table(table)}
                RETAIN {retention_hours} HOURS DRY RUN
                """
            ).count()
            print(
                f"{table}: VACUUM dry run found {eligible} files eligible "
                f"under the {retention_hours}-hour policy."
            )
        elif vacuum_mode == "EXECUTE":
            spark.sql(
                f"VACUUM {quoted_table(table)} RETAIN {retention_hours} HOURS"
            ).collect()
            print(f"{table}: VACUUM executed with {retention_hours} hours.")
        else:
            print(f"{table}: VACUUM skipped.")

    metrics_df = spark.createDataFrame(metrics, metric_schema)
    metrics_df.write.format("delta").mode("append").saveAsTable(metrics_table)

    missing_optimize = 0
    for table, _, _ in targets:
        optimize_count = (
            spark.sql(f"DESCRIBE HISTORY {quoted_table(table)}")
            .filter(F.col("operation") == "OPTIMIZE")
            .count()
        )
        missing_optimize += 0 if optimize_count else 1
    if missing_optimize:
        raise RuntimeError("OPTIMIZE evidence missing for one or more targets")

    upsert_audit(
        catalog, pipeline_run_id, pipeline_name, "optimize_zorder", start_ts,
        "SUCCESS", source_tables, job_run_id,
        end_ts=datetime.now(timezone.utc), rows_read=0,
        rows_written=len(metrics), rows_quarantined=0,
    )
except Exception as exc:
    upsert_audit(
        catalog, pipeline_run_id, pipeline_name, "optimize_zorder", start_ts,
        "FAILED", source_tables, job_run_id,
        end_ts=datetime.now(timezone.utc), error_message=str(exc)[:4000],
    )
    raise

spark.table(metrics_table).filter(
    F.col("pipeline_run_id") == pipeline_run_id
).orderBy("table_name", "metric_stage").display()
